In [1]:
!pip install wordfreq
!pip install sentence-transformers==2.2.2
import re
from scipy.stats import zscore
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import re
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, models, util
from nltk.corpus import stopwords
from wordfreq import top_n_list
from nltk.corpus import stopwords

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... - \ done
  Created wheel for sentence-transformers: filename=sentence_transformers-2.2.2-py3-none-any.whl size=125923 sha256=abcf7db01c7f523c663fca76fa62bd5cb26a54c8f2cf208cfde31be05db617d4
  Stored in directory: /root/.cache/pip/wheels/62/f2/10/1e606fd5f02395388f74e7462910fe851042f97238cbbd902f
Successfully built sentence-transformers


In [2]:
def get_embeddings(text, model):
    text = [str(t) for t in text]
    
    # Check for OOV words during tokenization
    tokenizer = model.tokenizer
    oov_words = []
    for word in text:
        tokens = tokenizer.tokenize(word)
        if "[UNK]" in tokens:
            oov_words.append(word)
    
    if oov_words:
        print(f"Warning: The following words are not in the model's vocabulary (OOV): {oov_words}")
    
    # Encode text in batches
    corpus_embeddings = model.encode(text, batch_size=1024, show_progress_bar=False, convert_to_tensor=True)
    assert len(corpus_embeddings) == len(text)
    return corpus_embeddings
    

def length_adjustment_bin(df, length_column, model_str, minimum_length=0):
    bins = range(minimum_length, df[length_column].max()+10, 10)
    df[f'{length_column}_bin'] = pd.cut(df[length_column], bins=bins)
    df[f'evidence_mean_{model_str}'] = df.groupby(f'{length_column}_bin')[f'evidence_score_{model_str}'].transform('mean')
    df[f'evidence_adj_{model_str}'] = df[f'evidence_score_{model_str}'] - df[f'evidence_mean_{model_str}']
    df[f'intuition_mean_{model_str}'] = df.groupby(f'{length_column}_bin')[f'intuition_score_{model_str}'].transform('mean')
    df[f'intuition_adj_{model_str}'] = df[f'intuition_score_{model_str}'] - df[f'intuition_mean_{model_str}']
    return df
    
def evidence_minus_intuition_score(df, model_str):
    df[[f'evidence_z_{model_str}', f'intuition_z_{model_str}']] = df[[f'evidence_adj_{model_str}', f'intuition_adj_{model_str}']].apply(zscore)
    df[f'emi_{model_str}'] = df[f'evidence_z_{model_str}'] - df[f'intuition_z_{model_str}']
    return df


In [3]:
def main(df):
    tqdm.pandas()

    df = df.dropna(subset=['text_clean'])
    df['text_clean'].astype(str)
    df = df[df['text_clean'].str.strip() != '']
    print(len(df))
    
    evidence_keywords = pd.read_csv("/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv") 
    evidence_keywords = list(evidence_keywords['evidence'])  
    
    intuition_keywords = pd.read_csv("/kaggle/input/dictionary2/PRODEMINFO_German_keywords.csv") 
    intuition_keywords = list(intuition_keywords['intuition'][0:38])  

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = SentenceTransformer("/kaggle/input/sbert-twitter/twitter_model")
    model_str = "w2vparliament"
    evidence_sim = torch.Tensor().to(torch.device(device))
    intuition_sim = torch.Tensor().to(torch.device(device))


    chunk_size = 10
    list_df = [df[idx:idx+chunk_size] for idx in range(0, len(df), chunk_size)]
    for batch in tqdm(list_df):
        batch_text = batch['text_clean']
        batch_text = list(batch_text)
        text_embeddings = get_embeddings(batch_text, model)       

       
        evidence_embeddings = get_embeddings(evidence_keywords, model)
        evidence_embeddings = torch.mean(evidence_embeddings, dim=0).to(device)


        intuition_embeddings = get_embeddings(intuition_keywords, model)
        intuition_embeddings = torch.mean(intuition_embeddings, dim=0).to(device)

        evidence_sim = torch.cat((evidence_sim, util.cos_sim(text_embeddings, evidence_embeddings)), 0)
        intuition_sim = torch.cat((intuition_sim, util.cos_sim(text_embeddings, intuition_embeddings)), 0)
        torch.cuda.empty_cache()
    
    print(len(df))
    print(len(evidence_sim.cpu().numpy()))
    df[f'evidence_score_{model_str}'] = evidence_sim.cpu().numpy()
    df[f'intuition_score_{model_str}'] = intuition_sim.cpu().numpy()

    length_column = 'length'
    df = length_adjustment_bin(df, length_column, model_str,0)
    df = evidence_minus_intuition_score(df, model_str) 

    
    model = SentenceTransformer("/kaggle/input/fasttext-sbert/fasttext_model")
    model_str = "w2vfasttext"
    evidence_sim = torch.Tensor().to(torch.device(device))
    intuition_sim = torch.Tensor().to(torch.device(device))

    for batch in tqdm(list_df):
        batch_text = batch['text_clean']
        batch_text = list(batch_text)
        text_embeddings = get_embeddings(batch_text, model)       

        evidence_embeddings = get_embeddings(evidence_keywords, model)
        evidence_embeddings = torch.mean(evidence_embeddings, dim=0).to(device)

        intuition_embeddings = get_embeddings(intuition_keywords, model)
        intuition_embeddings = torch.mean(intuition_embeddings, dim=0).to(device)

        evidence_sim = torch.cat((evidence_sim, util.cos_sim(text_embeddings, evidence_embeddings)), 0)
        intuition_sim = torch.cat((intuition_sim, util.cos_sim(text_embeddings, intuition_embeddings)), 0)     
        torch.cuda.empty_cache()

    df[f'evidence_score_{model_str}'] = evidence_sim.cpu().numpy()
    df[f'intuition_score_{model_str}'] = intuition_sim.cpu().numpy()

    length_column = 'length'
    df = length_adjustment_bin(df, length_column, model_str, 0)
    df = evidence_minus_intuition_score(df, model_str) 
    print(len(df))
    df.to_csv("twitter.csv", index = False)

In [4]:
df = pd.read_csv("/kaggle/input/final-data/twitter.csv")

main(df)

921676


100%|██████████| 92168/92168 [04:33<00:00, 336.97it/s]
/tmp/ipykernel_24/1679322505.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'evidence_mean_{model_str}'] = df.groupby(f'{length_column}_bin')[f'evidence_score_{model_str}'].transform('mean')
/tmp/ipykernel_24/1679322505.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'intuition_mean_{model_str}'] = df.groupby(f'{length_column}_bin')[f'intuition_score_{model_str}'].transform('mean')


921676
921676


100%|██████████| 92168/92168 [04:31<00:00, 339.88it/s]
/tmp/ipykernel_24/1679322505.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'evidence_mean_{model_str}'] = df.groupby(f'{length_column}_bin')[f'evidence_score_{model_str}'].transform('mean')
/tmp/ipykernel_24/1679322505.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[f'intuition_mean_{model_str}'] = df.groupby(f'{length_column}_bin')[f'intuition_score_{model_str}'].transform('mean')


921676
